# Field Context Lifecycle Hybrid RAG Demo

This notebook is the connectivity-resume scenario, not just a list of resources. It starts from a deliberately cleaned and limited local vector store, records the worker's search history, simulates connectivity returning, pulls only relevant online cases, then performs another offline search to prove the local context was enriched.

## Information flow

```mermaid
flowchart LR
  Clean[Clean local vector store] --> Limited[Seed only 3 offline cases]
  Limited --> SearchHistory[Offline search history]
  SearchHistory -->|connectivity resumes| Online[Azure AI Search full enriched index]
  Online --> Select[Select relevant online-only cases]
  Select --> Delta[SQLite offline delta pack]
  Delta --> Later[Later offline search uses seed + delta]
  Later --> Phi[Phi-4-mini drafts cited response]
```


## Initial offline context is intentionally limited

The demo cleans the local SQLite vector store and indexes only these cases: `INC-001, INC-002, INC-005`. This mimics a constrained device pack where only frequent/high-priority scenarios are cached before the worker enters a low-connectivity area.

After connectivity resumes, the demo syncs these online-only cases into the local delta pack: `ONL-007, ONL-010, ONL-011`. The local store then contains `6` cases.

## Limited offline -> online resume -> later offline result

| Search-history scenario | Initial offline top | Online top after connectivity resumes | Synced into offline delta | Later offline top |
| --- | --- | --- | --- | --- |
| Known concrete defect already cached | INC-002 | INC-002 | none | INC-002 |
| Water ingress becomes electrical safety issue | INC-001 | ONL-007 | ONL-007 | ONL-007 |
| Temporary works prop not in local pack | INC-005 | ONL-010 | ONL-010 | ONL-010 |
| Confined space gas alarm not in local pack | INC-005 | ONL-011 | ONL-011 | ONL-011 |

## Interpretation

The first offline search can answer common cases such as concrete honeycombing, but it misses richer context combinations. For water plus temporary electrical equipment, temporary works deformation, and confined-space gas alarms, the online path retrieves a more specific incident. Those specific incidents are cached and become available for the next offline search.

## Synced context records

### ONL-007 - Water ingress approaching temporary electrical riser

**Why cached:** Retain as an online-enriched incident because it benefits from live-context examples of temporary power exposure, but it also serves as a high-priority safety retrieval record when connectivity is available.

**Root cause:** Likely failed waterproofing, rainwater intrusion, or uncontrolled surface drainage compounded by insufficient protection and routing of temporary electrical infrastructure.

1. Isolate the affected temporary electrical circuit and verify de-energization by a competent person.
1. Keep personnel away from the wet area and establish a controlled exclusion zone.
1. Identify the water source and apply temporary diversion, containment, or pumping as needed.
1. Inspect the riser for ingress, damaged insulation, and elevated termination points.
1. Restore power only after the area is dry, the source is controlled, and electrical integrity is confirmed.

**Escalation:** Escalate immediately to electrical supervisor and site safety if live equipment is threatened, if any arc/odor/heat is observed, or if isolation cannot be verified.

### ONL-010 - Temporary works prop shows visible deformation

**Why cached:** Retain as an online-enriched record because temporary works deformation is a high-consequence event where current site conditions and recent load history materially affect interpretation.

**Root cause:** Possible overload from premature loading, inadequate bracing, incorrect prop capacity selection, or base settlement on an insufficient bearing surface.

1. Immediately clear personnel from the supported area and prohibit loading on the temporary arrangement.
1. Inspect all related props, bracing, and bearing conditions for signs of instability or settlement.
1. Verify the approved temporary works design against actual loading and geometry.
1. Install additional supports only under the direction of the temporary works coordinator or engineer.
1. Record the defect and do not reinstate use until the system is recertified safe.

**Escalation:** Escalate immediately to the temporary works coordinator, construction manager, and safety lead; treat as a potential collapse precursor.

### ONL-011 - Confined-space gas detector alarm triggered during drainage inspection

**Why cached:** Retain as an online-enriched record because gas alarm events benefit from current contextual cues, permit status, and sensor-driven interpretation in a hybrid CV-RAG workflow.

**Root cause:** Likely accumulation of sewer gas, inadequate ventilation, residual biological decay, or disturbed sediment releasing hazardous vapors.

1. Withdraw all personnel from the confined space area and prevent re-entry.
1. Test atmosphere from outside the space using calibrated equipment at multiple levels.
1. Implement forced ventilation only if approved and appropriate for the hazard identified.
1. Confirm rescue arrangements, permit status, and competent supervision before any re-entry.
1. Investigate source conditions and revise the confined-space entry controls accordingly.

**Escalation:** Escalate immediately to confined-space rescue resources and site management if anyone is inside the space, if gas readings are unstable, or if there is any sign of distress.

## Full online search with more queries

| Online query | Top online result | Title |
| --- | --- | --- |
| water seepage near temporary electrical riser | ONL-007 | Water ingress approaching temporary electrical riser |
| falling object exclusion zone breach below overhead works | ONL-008 | Falling object exclusion zone breached beneath overhead work |
| concrete spalling at slab edge after formwork striking | ONL-009 | Concrete spalling at slab edge after formwork striking |
| temporary works prop deformation under slab load | ONL-010 | Temporary works prop shows visible deformation |
| confined space gas detector alarm before manhole entry | ONL-011 | Confined-space gas detector alarm triggered during drainage inspection |
| mobile crane boom operating near overhead service line | ONL-012 | Mobile crane lift conducted near overhead service line |
| MEP duct clashes with ceiling service zone | INC-004 | MEP duct coordination clash at ceiling void |
| crack near lift core wall opening after concrete placement | INC-006 | Hairline crack observed near lift core wall opening |

## Where Phi-4-mini helps in the hybrid scenario

Phi-4-mini should run after retrieval in both offline passes. Before sync, it should be conservative: cite the limited evidence, explain that the local pack lacks a specialist case, and escalate. After sync, it can use the newly cached online-only case to produce a much more specific response while still operating offline.

For example, once `ONL-010` is cached, a later disconnected temporary-works query can cite the prop-deformation case, warn not to adjust the prop without a temporary works engineer, and list the exact stop-work and inspection sequence from the cached evidence.

In [1]:
import json
from pathlib import Path

summary = json.loads(Path("assets/context_lifecycle/context_lifecycle_summary.json").read_text())
print(f"Initial offline IDs: {', '.join(summary['initial_offline_ids'])}")
print(f"Synced IDs after connectivity resumed: {', '.join(summary['synced_ids'])}")
print(f"Offline store size after sync: {summary['offline_after_sync_count']}")


Initial offline IDs: INC-001, INC-002, INC-005
Synced IDs after connectivity resumed: ONL-007, ONL-010, ONL-011
Offline store size after sync: 6
